In [ ]:
# 1. Instal·lació de dependències per a Google Colab
%%shell
wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google.list
apt-get update
apt-get install -y google-chrome-stable
pip install selenium==4.18.1 webdriver-manager
google-chrome-stable --version

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import quote
import glob
import os
import time
import shutil
import pandas as pd
from google.colab import files

In [ ]:
def web_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("--window-size=1920,1200")
    
    # Forcem la carpeta de descàrregues a /content/
    prefs = {"download.default_directory": "/content/"}
    options.add_experimental_option("prefs", prefs)
    
    # Driver setup
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver

In [ ]:
def descarrega_xlsx(nou_nom):
    # Busca tots els arxius xlsx a /content/ que NO siguin el fitxer d'entitats
    xlsx_files = glob.glob('/content/brand_db_export*.xlsx')
    if not xlsx_files:
         xlsx_files = [f for f in glob.glob('/content/*.xlsx') if f not in ['/content/entitats.xlsx', '/content/log_results.xlsx']]
    
    if not xlsx_files:
        return False

    # Agafa el més recent
    xlsx_files.sort(key=os.path.getmtime, reverse=True)
    ultim_arxiu = xlsx_files[0]

    wipo_folder = '/content/wipo'
    if not os.path.exists(wipo_folder):
        os.makedirs(wipo_folder)
    
    safe_name = str(nou_nom).replace('/', '_').replace('\\', '_').replace(':', '_').replace('*', '_')
    nou_nom_complet = os.path.join(wipo_folder, safe_name + ".xlsx")
    
    try:
        if os.path.exists(nou_nom_complet): os.remove(nou_nom_complet)
        shutil.move(ultim_arxiu, nou_nom_complet)
        return True
    except Exception as e:
        print(f"      Error al desar el fitxer: {e}")
        return False

In [ ]:
def titular(driver, codi):
  print(f"Processant: {codi}...")
  
  # 1. Carreguem la pàgina principal de cerca per inicialitzar la sessió
  # Això evita redireccions i pèrdues de dades des del mateix domini
  driver.get("https://branddb.wipo.int/en/advancedsearch")
  
  try:
    # 2. Esperem que l'aplicació carregui i busquem el camp de cerca (Filter)
    wait = WebDriverWait(driver, 30)
    
    # Busquem l'input de text principal on podem escriure el titular
    # Sol tenir placeholder 'Filter...' o similar
    selector_input = "input[placeholder*='Filter'], input[type='text'], input[role='combobox']"
    search_input = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector_input)))
    
    # 3. Escrivim el nom de l'entitat entre cometes per a cerca exacta
    search_input.clear()
    # Fem servir Keys.ENTER per disparar la cerca des de dins de l'app
    search_input.send_keys(f'"{codi}"')
    time.sleep(1) # Petit temps de delay pel buffer d'Angular
    search_input.send_keys(Keys.ENTER)

    # 4. Esperem la càrrega de resultats o la confirmació de BUIT
    download_xpath = '//span[text()="Download results"]'
    
    # Esperem fins que aparegui o bé el botó o bé el missatge de zero resultats
    wait.until(lambda d: d.find_elements(By.XPATH, download_xpath) or 
                         "Displaying 0-0" in d.page_source or 
                         "No records found" in d.page_source)
    
    if not driver.find_elements(By.XPATH, download_xpath):
        print(f"   ! Cap marca trobada per a {codi}")
        return codi, "No trobat"

    # Si hi ha resultats, procedim amb clics i descàrrega
    btn = driver.find_element(By.XPATH, download_xpath)
    driver.execute_script("arguments[0].scrollIntoView();", btn)
    btn.click()
    
    excel_opt = WebDriverWait(driver, 15).until(
        EC.element_to_be_clickable((By.XPATH, '//li//span[text()="EXCEL"]'))
    )
    excel_opt.click()

    # Temps d'espera per la descàrrega
    time.sleep(12)

    if descarrega_xlsx(codi):
        print(f"   OK: Descarregada informació de {codi}")
        return codi, "1"
    else:
        print(f"   ? Fitxer no trobat després de descàrrega per a {codi}")
        return codi, "Error fitxer"

  except TimeoutException:
    print(f"   Timeout intentant buscar: {codi}")
    return codi, "Timeout"
  except Exception as e:
    print(f"   Error inesperat amb {codi}: {type(e).__name__}")
    return codi, f"Error {type(e).__name__}"

In [ ]:
# Carreguem el fitxer entitats.xlsx (PUJA'L A COLAB primer!)
if os.path.exists("entitats.xlsx"):
    df = pd.read_excel("entitats.xlsx")
    # Neteja de noms per a la cerca
    entities = df['Titular (entitat)'].apply(lambda x: str(x).rsplit(",", 1)[0].rsplit(' (', 1)[0].strip()).unique()

    log = []
    print(f"Iniciant driver per a {len(entities)} entitats...")
    driver = web_driver()
    
    try:
        for ent in entities:
            if ent and ent != 'nan':
                entitat, status = titular(driver, ent)
                log.append([entitat, status])
    finally:
        driver.quit()
        pd.DataFrame(log, columns=["Entitat", "Estat"]).to_excel("log_results.xlsx", index=False)
        print("\nProcés finalitzat. Resultats a log_results.xlsx")
else:
    print("ERROR: Puja el fitxer 'entitats.xlsx' al menú de l'esquerra de Colab.")

In [ ]:
if os.path.exists('/content/wipo'):
    if os.path.exists('marques_wipo_export.zip'): os.remove('marques_wipo_export.zip')
    shutil.make_archive('marques_wipo_export', 'zip', '/content/wipo')
    files.download('marques_wipo_export.zip')
    print("Descàrrega de ZIP preparada.")